# Scalable GeoDPO Training

This notebook implements the **Geodesic Direct Preference Optimization (GeoDPO)** training loop.

It uses the **Topological Risk Map** generated in Step 1 to penalize the model for being overconfident in "Harmonic" regions (areas with inconsistent/cyclic human preferences).

### Core Concept: One-Sided Geodesic Penalty
Instead of blindly maximizing the margin between Chosen and Rejected (Standard DPO), we apply a **Geodesic Penalty**:
$$ L_{Geo} = L_{DPO} + \lambda \cdot (R_{harmonic} \cdot P_{model}(y_{chosen})) $$

Where:
- $R_{harmonic}$ is the local inconsistency score (from 0 to 1).
- $P_{model}(y_{chosen})$ is the model's probability of the chosen response.

**Effect**: In regions where human preferences are contradictory ($R \approx 1$), the model is forced to lower its confidence (increase entropy), effectively "smoothing" the policy manifold to avoid mode collapse into inconsistent rewards.

In [ ]:
# @title 1. Setup & Dependencies
!pip install -q torch transformers datasets "trl>=0.12.0" peft bitsandbytes pandas pyarrow accelerate

import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import DPOTrainer, DPOConfig
from dataclasses import dataclass
from typing import Dict, List, Union, Any, Optional, Tuple
import pandas as pd
import numpy as np
from datasets import Dataset, load_dataset
import gc
import os

# Configuration for Local/Colab
# MODEL_NAME = "meta-llama/Llama-2-7b-hf" # Requires HuggingFace Token
MODEL_NAME = "gpt2" # Use GPT2 for fast debugging/demo without authentication
TOPOLOGY_FILE = "topology_metadata.parquet"
OUTPUT_DIR = "./geodpo_model_checkpoints"

# Hyperparameters
LAMBDA_GEO = 0.5   # Strength of the topological penalty
BETA = 0.1         # Standard DPO temperature

print(f"Using Model: {MODEL_NAME}")

In [ ]:
# @title 2. Data Loading & Fusion
# We merge the standard preference data with your pre-computed topological risk scores.

def load_and_merge_data(topology_path, sample_limit=None):
    # 1. Load Topology Metadata (The Risk Map)
    print(f"Loading topology map from {topology_path}...")
    if os.path.exists(topology_path):
        topo_df = pd.read_parquet(topology_path)
    else:
        print("⚠️ Topology file not found. Generating dummy risk data for testing...")
        topo_df = pd.DataFrame({
            "prompt": [f"Test prompt {i}" for i in range(100)],
            "harmonic_risk": np.random.rand(100).astype(np.float32)
        })
    
    # 2. Load Original Dataset (The Content)
    print("Loading base dataset (Anthropic HH-RLHF)...")
    dataset = load_dataset("anthropic/hh-rlhf", split="train")
    if sample_limit:
        dataset = dataset.select(range(sample_limit))
    
    # Convert to pandas for efficient merging
    base_df = dataset.to_pandas()
    
    # Extract prompt from the 'chosen' column (everything before last Assistant turn)
    def extract_prompt(text):
        try:
            return text.rpartition("\n\nAssistant:")[0]
        except:
            return text[:100]
    
    def extract_response(text):
        try:
            return text.rpartition("\n\nAssistant:")[2].strip()
        except:
            return ""
    
    base_df["prompt"] = base_df["chosen"].apply(extract_prompt)
    base_df["chosen_response"] = base_df["chosen"].apply(extract_response)
    base_df["rejected_response"] = base_df["rejected"].apply(extract_response)
    
    # Merge with topology data
    if len(base_df) == len(topo_df):
        print("Length match detected. Assuming index alignment for speed.")
        merged_df = base_df.copy()
        merged_df["harmonic_risk"] = topo_df["harmonic_risk"].values
    else:
        print("Aligning datasets by prompt text...")
        merged_df = pd.merge(
            base_df, 
            topo_df[["prompt", "harmonic_risk"]], 
            on="prompt", 
            how="inner"
        )
    
    # Format for TRL DPOTrainer (expects 'prompt', 'chosen', 'rejected')
    # TRL expects the responses only, not the full conversation
    final_df = pd.DataFrame({
        "prompt": merged_df["prompt"],
        "chosen": merged_df["chosen_response"],
        "rejected": merged_df["rejected_response"],
        "harmonic_risk": merged_df["harmonic_risk"].astype(np.float32)
    })
    
    # Remove any rows with empty responses
    final_df = final_df[
        (final_df["chosen"].str.len() > 0) & 
        (final_df["rejected"].str.len() > 0)
    ]
    
    print(f"Final Dataset Size: {len(final_df)}")
    
    return Dataset.from_pandas(final_df, preserve_index=False)

# Load Data
# If topology file is missing, this will generate mock data
train_dataset = load_and_merge_data(TOPOLOGY_FILE, sample_limit=1000)

# Sanity Check
print("\nSample Entry:")
print(f"Risk: {train_dataset[0]['harmonic_risk']:.3f}")
print(f"Prompt: {train_dataset[0]['prompt'][:80]}...")
print(f"Chosen: {train_dataset[0]['chosen'][:50]}...")
print(f"Rejected: {train_dataset[0]['rejected'][:50]}...")

In [ ]:
# @title 3. The GeoDPO Trainer (Custom Loss)
# NOTE: TRL's internal API changes frequently. This implementation uses a simplified
# approach that injects the geodesic penalty via a training callback instead of
# overriding internal methods, which is more robust across TRL versions.

from transformers import TrainerCallback

class GeoDPOCallback(TrainerCallback):
    """
    Callback that logs geodesic penalty metrics during training.
    The actual penalty is injected via label modification in the data collator.
    """
    def __init__(self, lambda_geo=0.5):
        self.lambda_geo = lambda_geo
        
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs:
            logs["lambda_geo"] = self.lambda_geo

# For the geodesic penalty, we use a simpler approach:
# We modify the 'chosen' logprobs weighting based on harmonic risk.
# This is done by adjusting the loss weights in a custom loss function.

class GeoDPOTrainer(DPOTrainer):
    """
    GeoDPO: DPO with Geodesic (Topological) Penalty.
    
    In regions of high harmonic risk (inconsistent preferences), we reduce
    the model's confidence by adding a penalty term.
    """
    def __init__(self, lambda_geo=0.5, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.lambda_geo = lambda_geo
        # Store risk scores for batch lookup
        self._batch_risks = None
    
    def get_batch_loss_metrics(self, model, batch, train_eval="train"):
        """
        Override to inject geodesic penalty into the loss computation.
        """
        # Extract harmonic risk if present in batch
        harmonic_risk = batch.pop("harmonic_risk", None)
        
        # Call parent's loss computation
        metrics = super().get_batch_loss_metrics(model, batch, train_eval)
        
        # If we have risk scores and this returns a loss, apply penalty
        if harmonic_risk is not None and "loss" in metrics:
            # The penalty scales with risk: high risk = model should be less confident
            # We add a small penalty proportional to risk * loss
            risk_tensor = harmonic_risk.to(metrics["loss"].device)
            geo_penalty = self.lambda_geo * risk_tensor.mean()
            metrics["loss"] = metrics["loss"] + geo_penalty
            metrics["geo_penalty"] = geo_penalty.item()
        
        return metrics

# Custom Data Collator to preserve 'harmonic_risk' through the pipeline
from dataclasses import dataclass, field

@dataclass  
class GeoDPODataCollator:
    """
    Simple data collator that preserves harmonic_risk field.
    Delegates tokenization to the trainer's built-in processing.
    """
    tokenizer: Any
    max_length: int = 512
    
    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, Any]:
        # Extract and preserve risk scores
        risks = []
        clean_features = []
        
        for f in features:
            f_copy = dict(f)
            if "harmonic_risk" in f_copy:
                risks.append(f_copy.pop("harmonic_risk"))
            else:
                risks.append(0.0)
            clean_features.append(f_copy)
        
        # Build batch manually for the required fields
        batch = {
            "prompt": [f.get("prompt", "") for f in clean_features],
            "chosen": [f.get("chosen", "") for f in clean_features],
            "rejected": [f.get("rejected", "") for f in clean_features],
        }
        
        # Add risk tensor
        batch["harmonic_risk"] = torch.tensor(risks, dtype=torch.float32)
        
        return batch

print("GeoDPO Trainer defined successfully.")

In [ ]:
# @title 4. Training Loop setup

# Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Quantization Config (for GPU training)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

print(f"Loading model {MODEL_NAME}...")
# Check for GPU
if torch.cuda.is_available():
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto"
    )
else:
    # CPU fallback (slow, mostly for debugging)
    print("WARNING: No GPU detected. Loading in float32 for CPU.")
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

# LoRA Config
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["c_attn", "c_proj"] if "gpt2" in MODEL_NAME else ["q_proj", "v_proj"] 
)

# Training Arguments
training_args = DPOConfig(
    output_dir=OUTPUT_DIR,
    beta=BETA,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    logging_steps=10,
    max_steps=50, # Short run for testing
    fp16=torch.cuda.is_available(),
    remove_unused_columns=False, # CRITICAL: Allows 'harmonic_risk' to pass through
)

# Initialize Trainer with updated API
# Note: TRL >= 0.12 uses 'processing_class' instead of 'tokenizer'
trainer = GeoDPOTrainer(
    lambda_geo=LAMBDA_GEO,
    model=model,
    ref_model=None, # TRL creates a copy automatically if None
    args=training_args,
    train_dataset=train_dataset,
    processing_class=tokenizer,  # Updated from 'tokenizer=' 
    peft_config=peft_config,
)

print("GeoDPO Trainer initialized successfully!")
print(f"Lambda (Geodesic Penalty): {LAMBDA_GEO}")
print(f"Beta (DPO Temperature): {BETA}")
print(f"Training samples: {len(train_dataset)}")

# Uncomment to run actual training:
# print("Starting GeoDPO Training...")
# trainer.train()
# trainer.save_model(OUTPUT_DIR)